# 🫀 실험 20b — **환자 단위 집계**, 그리고 `AMI` 는 정말 `AMI` 인가

**MedKOS / `notebooks/exp20b_patient_level.ipynb`** · 퀘스트 `ailab-2026-0015` · **학습 0회**

실험20 이 남긴 저장 arm 만 다시 읽는다. GPU 불필요, 다운로드 없음(헤더 1MB 만).

## 왜 이 실험인가

실험20 은 **부위별** 숫자만 냈다. 그런데 임상 질문은 부위가 아니라 환자다 —
**"이 MI 환자를 놓쳤나"**, **"이 정상인에게 헛경보를 울렸나"**.
부위 라벨이 다중이라 **양성 라벨 161개 ≠ 환자 290명** 이므로 부위 표를 더해서는 안 나온다.

그리고 실험20 사후 유도에서 나온 "비MI 환자의 98.3% 가 최소 1개 경보" 는
**부위 예측이 독립이라는 가정**에서 나온 상한이다. 실제로 부위 예측은 서로 강하게
상관돼 있으므로 이 숫자는 틀렸다 — **얼마나 틀렸는지를 재는 것**이 이 실험이다.

마지막으로 `AMI` 붕괴(외부 0.5684, 12유도로도 0.6104)의 원인을 **직접** 가른다.

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **P-1** | 7부위 OR 의 환자 단위 특이도 `Sp_pt` | `Sp_pt < 0.50` — 비MI 절반 이상이 경보 |
| **P-2** | `AMI` 를 빼면 위양성 **오즈**가 얼마나 줄어드나 | `OR ≥ 1.5` 면 **한 부위가 OR 을 지배** |
| **P-3** | MI 환자 중 적어도 한 부위를 맞춘 비율 | `hit_partial ≥ 0.70` |
| **P-4★★** | PTBDB 에 **"anterior" 만 적힌** 환자를 `ASMI` 헤드가 `AMI` 헤드보다 잘 잡나 | 차 `> 0.05` → **라벨 정의 불일치 지지** |
| **P-5★★** | **이중차분** — `(ASMI−AMI)` 격차가 "anterior" 에서 "inferior" 보다 큰가 | 차 `> 0.05` → P-4 가 "ASMI 헤드가 원래 세서" 로 설명되지 않음 |

### 이 실험이 고치는 채점 결함

실험20 에서 **환자 2명짜리 부분군에 시드 CI 가 붙어 거짓 ✅** 가 떴다.
시드 CI 는 재학습 잡음만 담고 **환자 표집 잡음을 못 본다.** 그래서 여기서는

- **`GMIN_SUB = 20`** — 부분군도 20명 미만이면 채점에서 뺀다(부위에만 걸었던 걸 한 층 내린다)
- **시드 t-CI 와 환자 부트스트랩 CI 를 나란히** 낸다. 둘 중 넓은 쪽이 진짜 불확실성이다

### 하지 않는 것

- 임계값 재조정 (내부에서 동결한 것을 그대로 쓴다 — 재조정하면 외부검증이 아니다)
- 새 학습 · 새 구성 · 새 전처리


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정 (학습 0회 · GPU 불필요)
!pip -q install wfdb

import os, sys, json, time, re, subprocess, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~22 와 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
LEADS = {"I+II+V2+V5": [0, 1, 7, 10], "12": list(range(12))}
# ★★ 여기까지

DEPLOY, REF = "I+II+V2+V5", "12"
SEEDS = [0, 1, 2]
GMIN_PT, GMIN_SUB = 20, 20     # ★ 부분군에도 최소 n 을 건다(실험20 에서 빠뜨린 것)
SENS_TARGET = 0.90
SP_THR, OR_THR, HIT_THR, HEAD_THR = 0.50, 1.5, 0.70, 0.05
# ★ P-2 를 절대 ΔSp 로 재면 안 된다 — 특이도가 0 근처면 천장에 막혀 아무리 큰 효과도
#   작게 나온다(픽스처 ④ 가 잡았다). 실험22 에서 위험비 → 오즈비로 바꾼 것과 같은 이유.
BOOT = 2000
PTBDB_URL = "https://physionet.org/files/ptbdb/1.0.0"

CONFIG = dict(exp="exp20b_patient_level", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp18_confirm", "exp19_two_stage"],
              purpose=("부위별이 아니라 **환자 단위**로 잰다. 그리고 AMI 붕괴가 "
                       "모델 실패인지 라벨 정의 불일치인지 직접 가른다"),
              dataset="PTB Diagnostic ECG Database (PhysioNet, Open Access)",
              change_one_thing="새 학습·새 임계값 없음. 실험20 의 저장 arm 재집계",
              deploy=DEPLOY, seeds=SEEDS, sens_target=SENS_TARGET,
              gmin_patients=GMIN_PT, gmin_subgroup=GMIN_SUB, boot=BOOT,
              ci_rule=("시드 t-CI **와** 환자 부트스트랩 CI 를 나란히 낸다. 실험20 에서 "
                       "환자 2명 부분군에 시드 CI 가 붙어 거짓 지지가 떴다 — 시드 CI 는 "
                       "재학습 잡음만 담고 환자 표집 잡음을 못 본다"),
              predictions={
                  "P-1": f"7부위 OR 의 환자 단위 특이도 < {SP_THR}",
                  "P-2": f"AMI 를 빼면 위양성 오즈비 >= {OR_THR}",
                  "P-3": f"MI 환자 중 적어도 한 부위 적중 >= {HIT_THR}",
                  "P-4": (f"PTBDB 'anterior' 환자에서 ASMI 헤드 AUROC − AMI 헤드 AUROC "
                          f"> {HEAD_THR} → 라벨 정의 불일치 지지"),
                  "P-5": ("이중차분 — (ASMI−AMI) 격차가 'anterior' 에서 'inferior' 보다 "
                          f"{HEAD_THR} 이상 크다. ASMI 헤드는 내부에서 원래 제일 세므로"
                          " (0.9505 vs AMI 0.8309) 이 대조가 없으면 P-4 를 해석할 수 없다")},
              caveat=("환자 단위 집계는 예측=평균·라벨=any. 임계값은 내부에서 동결한 것을 "
                      "그대로 쓴다(재조정하면 외부검증이 아니다)"))
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp20b_patient", CONFIG, project=PROJECT)

# ── 부모 실행 찾기. 실험20 이 **반드시** 있어야 한다(외부 arm 이 거기 있다)
REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if os.path.isdir(r.get("dir", "")):
        DIRS[r.get("exp_id")] = r["dir"]          # 같은 exp_id 는 최신이 이긴다
if "exp20_ptbdb" not in DIRS:
    raise RuntimeError(
        "실험20 실행을 registry 에서 못 찾았다 — 외부 예측 arm 이 거기 있다.\n"
        f"  registry: {REG}\n  발견된 실험: {sorted(DIRS)}")
D20 = DIRS["exp20_ptbdb"]
run.log(f"실험20 실행 디렉터리: {D20}")
# 내부 OOF arm 은 실험16·17·18·19 중 어디에 있을 수 있다 → 전부 뒤진다
PARENTS = [DIRS[k] for k in ("exp19_two_stage", "exp18_confirm",
                             "exp17_wearable", "exp16_four_lead") if k in DIRS]
run.log("내부 arm 후보: " + (", ".join(PARENTS) or "없음"))

def arm_at(d, n):
    p = os.path.join(d, "arms", n, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

def find_arm(n):
    for d in PARENTS:
        a = arm_at(d, n)
        if a is not None:
            return a
    return None

SITES18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                         encoding="utf-8"))["sites"]
run.log(f"부위 순서(arm 열과 일치해야 한다): {SITES18}")


In [ ]:
# CELL 2 — 라벨 재구성 (헤더 1MB · 캐시되면 즉시)
#   실험20 은 라벨을 파일로 안 남겼다. 헤더에서 다시 만들되 **한 번만** 만들고
#   `data/` 에 캐시한다(실험24 등에서도 쓴다).
import pandas as pd, wfdb

LBL_CACHE = run.data("ptbdb_labels_v1.json")
PDB = "/content/ptbdb"

def norm_loc(s):
    return re.sub(r"[^a-z]", "", str(s).strip().lower())

def split_locs(s):
    """통짜 정규화 — 구분자로 쪼개지 않는다('n/a' 가 'n'+'a' 로 갈라지는 것 방지)."""
    return [norm_loc(s)]

# ★ 실험20 CELL 1 과 **동일**해야 한다. 다르면 라벨이 달라져 비교가 무의미해진다.
LOC_MAP = {"anterior": ["AMI"], "anteroseptal": ["ASMI"], "anteriorseptal": ["ASMI"],
           "anterolateral": ["ALMI"], "anteriorlateral": ["ALMI"],
           "anteroapicallateral": ["ALMI"], "anteroseptallateral": ["ASMI"],
           "anteroseptolateral": ["ASMI"], "inferior": ["IMI"],
           "inferolateral": ["ILMI"], "inferiorlateral": ["ILMI"],
           "inferoposterolateral": ["IPLMI"], "inferoposterlateral": ["IPLMI"],
           "inferiorposteriorlateral": ["IPLMI"], "lateral": ["LMI"],
           "inferolatera": ["ILMI"], "anteriorinferior": ["AMI", "IMI"],
           "anterioranterior": ["AMI"], "inferoposteriorinferior": ["IMI"]}
LOC_DROP = {"no", "nein", "unknown", "", "none", "na", "inferoposterior",
            "inferiorposterior", "posterior", "posterolateral", "posteriorlateral"}

def sites_of(v):
    out = []
    for k in split_locs(v):
        out += LOC_MAP.get(k, [])
    return sorted(set(out))

if os.path.exists(LBL_CACHE):
    H = pd.DataFrame(json.load(open(LBL_CACHE, encoding="utf-8")))
    run.log(f"라벨 캐시 적중 {LBL_CACHE} · {len(H)}건")
else:
    os.makedirs(PDB, exist_ok=True)
    recs_f = os.path.join(PDB, "RECORDS")
    if not (os.path.exists(recs_f) and os.path.getsize(recs_f) > 0):
        subprocess.run(["wget", "-q", "-O", recs_f, f"{PTBDB_URL}/RECORDS"], check=True)
    RECS = [x.strip() for x in open(recs_f) if x.strip()]
    need = [r for r in RECS if not os.path.exists(os.path.join(PDB, r + ".hea"))]
    if need:
        run.log(f"헤더 {len(need)}개 내려받는 중 (약 1MB)…")
        t0 = time.time()
        for i, r in enumerate(need):
            os.makedirs(os.path.join(PDB, os.path.dirname(r)), exist_ok=True)
            subprocess.run(["wget", "-q", "-O", os.path.join(PDB, r + ".hea"),
                            f"{PTBDB_URL}/{r}.hea"], check=False)
            if (i + 1) % 150 == 0:
                run.log(f"  {i+1}/{len(need)} · {time.time()-t0:.0f}s")
    rows = []
    for r in RECS:
        h = wfdb.rdheader(os.path.join(PDB, r))
        d = {"rec": r, "patient": r.split("/")[0]}
        for c in (h.comments or []):
            if ":" in c:
                k, v = c.split(":", 1)
                d[k.strip().lower()] = v.strip()
        rows.append(d)
    H = pd.DataFrame(rows)
    json.dump(H.fillna("").to_dict("records"), open(LBL_CACHE, "w", encoding="utf-8"),
              ensure_ascii=False)
    run.log(f"라벨 캐시 생성 {LBL_CACHE} · {len(H)}건")

ACOL = next(c for c in H.columns if c.startswith("acute infarction"))
FCOL = next(c for c in H.columns if c.startswith("former infarction"))
H["raw_a"] = H[ACOL].fillna("").map(norm_loc)
H["raw_f"] = H[FCOL].fillna("").map(norm_loc)
unmapped = sorted({k for k in set(H.raw_a) | set(H.raw_f)
                   if k not in LOC_MAP and k not in LOC_DROP})
if unmapped:
    raise LabelVocabError(f"매핑되지 않은 국소화 문자열 {unmapped} — 실험20 과 매핑이 어긋났다")
H["site_acute"] = H[ACOL].fillna("").apply(sites_of)
H["site_former"] = H[FCOL].fillna("").apply(sites_of)
H["sites"] = [sorted(set(a) | set(f)) for a, f in zip(H.site_acute, H.site_former)]
run.log(f"라벨 준비 · 레코드 {len(H)} · 환자 {H.patient.nunique()}명 · 미매핑 0건 ✅")

# ── arm 의 행 순서 = 실험20 신호 캐시의 recs 순서. 반드시 이걸로 정렬한다.
SIG = run.data("ptbdb_12lead_100hz_v2.npz")
if not os.path.exists(SIG):
    SIG = run.data("ptbdb_12lead_100hz.npz")     # v1 로 돌렸다면
if not os.path.exists(SIG):
    raise RuntimeError("PTBDB 신호 캐시가 없다 — arm 의 행 순서를 알 수 없다")
RKEEP = [str(x) for x in np.load(SIG, allow_pickle=True)["recs"]]
run.log(f"레코드 순서 기준: {os.path.basename(SIG)} · {len(RKEEP)}건")

HK = H.set_index("rec").loc[RKEEP]
YE = np.stack([[s in row for s in SITES18] for row in HK.sites]).astype(bool)
PT = HK.patient.values
RAW_A, RAW_F = HK.raw_a.values, HK.raw_f.values
run.log(f"외부 라벨 {YE.shape} · 환자 {len(set(PT))}명")


In [ ]:
# CELL 3 — 외부 arm 적재 + 환자 단위 집계 + **내부에서 동결한 임계값**
PE = {}
for sd in SEEDS:
    a = arm_at(D20, f"ext_{DEPLOY}_s{sd}")
    if a is None:
        raise RuntimeError(f"실험20 의 외부 arm ext_{DEPLOY}_s{sd} 가 없다: {D20}/arms")
    assert_arm_shape(a, len(RKEEP), name=f"ext_{DEPLOY}_s{sd}")
    PE[sd] = a
run.log(f"외부 arm {len(PE)}개 적재 · 모양 {PE[SEEDS[0]].shape}")

PTS = sorted(set(PT))
PIDX = {p: np.where(PT == p)[0] for p in PTS}
def to_patient(v, how="mean"):
    """★ 예측은 평균, 라벨은 any — 실험20 과 동일한 규약."""
    if how == "mean":
        return np.array([v[PIDX[p]].mean() for p in PTS])
    return np.array([v[PIDX[p]].any() for p in PTS])

Y_PT = np.stack([to_patient(YE[:, j], "any") for j in range(len(SITES18))], axis=1)
S_PT = {sd: np.stack([to_patient(PE[sd][:, j]) for j in range(len(SITES18))], axis=1)
        for sd in SEEDS}
IS_MI = Y_PT.any(axis=1)
run.log(f"환자 {len(PTS)}명 · MI {IS_MI.sum()}명 ({IS_MI.mean():.1%}) · "
        f"비MI {(~IS_MI).sum()}명")

# ── 내부(PTB-XL 5겹 OOF)에서 민감도 0.90 을 주는 임계값을 **동결**한다
PX = run.data("ptbxl_12lead_all.npz")
zx = np.load(PX, allow_pickle=True)
FOLD10, EID = zx["fold"], zx["eid"]
CV = (FOLD10 - 1) % K_FOLD
import ast, pandas as pd
csv = "/content/ptbxl/ptbxl_database.csv"
if not os.path.exists(csv):
    os.makedirs("/content/ptbxl", exist_ok=True)
    subprocess.run(["wget", "-q", "-O", csv,
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"], check=True)
dfa = pd.read_csv(csv, index_col="ecg_id").loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
assert_label_vocab(SITES18, {c for cs in dfa.codes for c in cs}, kind="MI 부위 코드")
YI = np.stack([[s in c for s in SITES18] for c in dfa.codes]).astype(bool)

def thr_sens(score, pos, t=SENS_TARGET):
    p = score[pos]
    return float(np.quantile(p, 1.0 - t, method="lower")) if len(p) else -np.inf

THR, SPEC_INT = {}, {}
for sd in SEEDS:
    th, sp = [], []
    for j in range(len(SITES18)):
        oof = np.zeros(len(EID), "float32")
        for k in range(K_FOLD):
            a = find_arm(f"{DEPLOY}_s{sd}_f{k}")
            if a is None:
                raise RuntimeError(
                    f"내부 OOF arm {DEPLOY}_s{sd}_f{k} 를 못 찾았다.\n"
                    f"  뒤진 곳: {PARENTS}\n"
                    "  → 임계값을 내부에서 동결할 수 없으면 이 실험은 성립하지 않는다.")
            oof[np.where(CV == k)[0]] = a[:, j]
        yi = YI[:, j]
        t_ = thr_sens(oof, yi)
        th.append(t_); sp.append(float((oof[~yi] < t_).mean()))
    THR[sd] = np.array(th); SPEC_INT[sd] = np.array(sp)
run.log("\n내부에서 동결한 임계값(민감도 0.90) · 그때의 내부 특이도")
run.log(f"  {'부위':<8}" + "".join(f"{'시드'+str(s):>10}" for s in SEEDS) + f"{'내부 특이도':>12}")
for j, s in enumerate(SITES18):
    run.log(f"  {s:<8}" + "".join(f"{THR[sd][j]:>10.4f}" for sd in SEEDS)
            + f"{np.mean([SPEC_INT[sd][j] for sd in SEEDS]):>12.3f}")

ALARM = {sd: (S_PT[sd] >= THR[sd][None, :]) for sd in SEEDS}


In [ ]:
# CELL 4 — 【P-1·P-2】 환자 단위 민감도·특이도·PPV (7부위 OR vs AMI 제외 6부위)
from scipy import stats

def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def stats_of(alarm_any, is_mi):
    se = float(alarm_any[is_mi].mean()) if is_mi.any() else np.nan
    sp = float((~alarm_any[~is_mi]).mean()) if (~is_mi).any() else np.nan
    ppv = float(is_mi[alarm_any].mean()) if alarm_any.any() else np.nan
    npv = float((~is_mi[~alarm_any]).mean()) if (~alarm_any).any() else np.nan
    return se, sp, ppv, npv

def boot_ci(fn, n, B=BOOT, seed=SEED0, conf=.95):
    """★ **환자**를 재표본한다. 시드 CI 가 못 보는 축이 이것이다.
    같은 재표본 인덱스를 모든 시드가 공유한다(짝지은 비교)."""
    vals = [fn(ix) for ix in boot_indices(n, B, seed)]
    vals = np.array([v for v in vals if np.isfinite(v)])
    lo, hi = np.percentile(vals, [(1 - conf) / 2 * 100, (1 + conf) / 2 * 100])
    return float(lo), float(hi)

AMI_J = SITES18.index("AMI") if "AMI" in SITES18 else None
KEEP6 = [j for j in range(len(SITES18)) if j != AMI_J]
VIEWS = {"7부위 OR": list(range(len(SITES18))), "AMI 제외 6부위 OR": KEEP6}

run.log("\n" + "=" * 112)
run.log("【P-1·P-2】 환자 단위 — 부위 판별기를 OR 로 묶으면 무슨 일이 나나")
run.log("=" * 112)
run.log(f"  환자 {len(PTS)}명 · MI {IS_MI.sum()}명 · 임계값은 내부 동결(민감도 0.90 목표)")
run.log(f"\n  {'구성':<20}{'Se':>8}{'Sp':>8}{'PPV':>8}{'NPV':>8}"
        f"{'경보 환자':>10}{'환자당 경보수':>14}")
PL = {}
for name, cols in VIEWS.items():
    per_seed = [stats_of(ALARM[sd][:, cols].any(axis=1), IS_MI) for sd in SEEDS]
    se, sp, ppv, npv = [np.mean([p[i] for p in per_seed]) for i in range(4)]
    rate = np.mean([ALARM[sd][:, cols].any(axis=1).mean() for sd in SEEDS])
    per_pt = np.mean([ALARM[sd][:, cols].sum(axis=1).mean() for sd in SEEDS])
    PL[name] = {"se": se, "sp": sp, "ppv": ppv, "npv": npv,
                "sp_seeds": [p[1] for p in per_seed], "cols": cols,
                "alarm_rate": float(rate), "alarms_per_patient": float(per_pt)}
    run.log(f"  {name:<20}{se:>8.3f}{sp:>8.3f}{ppv:>8.3f}{npv:>8.3f}"
            f"{rate:>10.1%}{per_pt:>14.2f}")

# ── 두 축의 CI 를 나란히 낸다(이게 이 실험의 방법론 목적이다)
run.log("\n  【CI 두 축】 시드 t-CI 는 재학습 잡음 · 환자 부트스트랩은 표집 잡음")
for name in VIEWS:
    cols = PL[name]["cols"]
    m, lo, hi = t_ci(PL[name]["sp_seeds"])
    def f(ix, cols=cols):
        mi = IS_MI[ix]
        return np.mean([float((~ALARM[sd][ix][:, cols].any(axis=1)[~mi]).mean())
                        for sd in SEEDS]) if (~mi).any() else np.nan
    blo, bhi = boot_ci(f, len(PTS))
    PL[name]["sp_tci"] = [lo, hi]; PL[name]["sp_bootci"] = [blo, bhi]
    run.log(f"  {name:<20} Sp {m:.3f} · 시드 [{lo:+.3f},{hi:+.3f}] 폭 {hi-lo:.3f}"
            f" · 환자부트 [{blo:.3f},{bhi:.3f}] 폭 {bhi-blo:.3f}"
            f"  → {'환자 표집' if (bhi-blo) > (hi-lo) else '시드'} 쪽이 넓다")

def fp_odds(cols):
    """비MI 환자의 **위양성 오즈**. 특이도 차이는 천장에 막히지만 오즈는 안 막힌다."""
    f = np.mean([float(ALARM[sd][:, cols].any(axis=1)[~IS_MI].mean()) for sd in SEEDS])
    return f / max(1 - f, 1e-9), f

o7, f7 = fp_odds(VIEWS["7부위 OR"])
o6, f6 = fp_odds(VIEWS["AMI 제외 6부위 OR"])
run.log(f"\n  위양성 오즈 — 7부위 {o7:.2f}(FPR {f7:.1%}) · 6부위 {o6:.2f}(FPR {f6:.1%})"
        f" · **오즈비 {o7 / max(o6, 1e-9):.2f}**")

# ── 독립 가정 상한과 실측의 거리 (사후 유도가 얼마나 틀렸나)
sp_site = np.array([np.mean([float((~ALARM[sd][:, j][~IS_MI]).mean()) for sd in SEEDS])
                    for j in range(len(SITES18))])
indep = 1 - float(np.prod(sp_site))
run.log(f"\n  독립 가정 상한: 비MI 의 {indep:.1%} 가 경보 · "
        f"실측: {1 - PL['7부위 OR']['sp']:.1%}")
run.log(f"  → 상한이 실측보다 {indep - (1 - PL['7부위 OR']['sp']):+.1%}p 과장돼 있다"
        " (부위 예측이 서로 상관돼 있다는 뜻)")


In [ ]:
# CELL 5 — 【P-3】 1단계를 통과한 MI 환자에서 부위를 맞추나
run.log("\n" + "=" * 112)
run.log("【P-3】 부위 적중 — MI 환자 중 적어도 한 부위를 맞춘 비율")
run.log("=" * 112)
HIT = {}
for name, cols in VIEWS.items():
    ex, pa = [], []
    for sd in SEEDS:
        A, Y = ALARM[sd][:, cols], Y_PT[:, cols]
        ex.append(float(((A == Y).all(axis=1))[IS_MI].mean()))
        pa.append(float(((A & Y).any(axis=1))[IS_MI].mean()))
    HIT[name] = {"exact": ex, "partial": pa}
    m, lo, hi = t_ci(pa)
    run.log(f"  {name:<20} 부분적중 {m:.3f} [{lo:.3f},{hi:.3f}] · "
            f"완전일치 {np.mean(ex):.3f}")
def f_hit(ix):
    mi = IS_MI[ix]
    if not mi.any():
        return np.nan
    return np.mean([float(((ALARM[sd][ix] & Y_PT[ix]).any(axis=1))[mi].mean())
                    for sd in SEEDS])
hlo, hhi = boot_ci(f_hit, len(PTS))
run.log(f"  환자 부트스트랩(7부위) [{hlo:.3f},{hhi:.3f}]")
run.log("  ※ 완전일치는 '켜야 할 부위만 정확히 켰나' 라 다중라벨에서 매우 엄격하다")


In [ ]:
# CELL 6 — 【P-4·P-5】 `AMI` 는 정말 `AMI` 인가 — 라벨 정합 직접 검정
#   가설: PTBDB 의 "anterior" 는 PTB-XL 이 ASMI 라고 부르는 것과 겹친다.
#   검정: "anterior 만 적힌" 환자를 **어느 헤드가 제일 잘 잡는지** 본다.
#         · 대조군은 MI 가 전혀 없는 환자로 공통 고정
#         · 헤드 간 확률 크기는 못 비교한다(교정이 헤드마다 다르다) → **AUROC** 로 잰다
#   ★ 함정: ASMI 헤드는 내부에서 원래 제일 세다(0.9505 vs AMI 0.8309). 그래서 P-4 만으로는
#     '라벨 불일치' 와 '헤드가 그냥 세다' 를 못 가른다 → **P-5 는 이중차분(DiD)** 이다.
from sklearn.metrics import roc_auc_score

SITESET = {p: set() for p in PTS}
for i_, p in enumerate(PT):
    SITESET[p] |= set(np.array(SITES18)[YE[i_]].tolist())

def pure_group(raw_key, only_site):
    """원문이 raw_key 이고 **그 부위 하나만** 달린 환자.

    다른 전벽 변종(ASMI·ALMI)이 함께 적힌 환자를 빼야 검정이 오염되지 않는다.
    """
    m = (RAW_A == raw_key) | (RAW_F == raw_key)
    hit = set(PT[m])
    return np.array([p in hit and SITESET[p] == {only_site} for p in PTS])

def head_profile(g, tag):
    """그룹 g 를 각 헤드가 얼마나 잘 잡는지(대조군 = 비MI 환자 공통)."""
    neg = ~IS_MI
    n = int(g.sum())
    run.log(f"\n  ── {tag} 환자 {n}명 vs 비MI {int(neg.sum())}명"
            + ("" if n >= GMIN_SUB else f"   ⚠️ {GMIN_SUB}명 미만 → 채점 제외"))
    if n == 0:
        return None
    prof = {}
    y = np.concatenate([np.ones(n, bool), np.zeros(int(neg.sum()), bool)])
    for j, s in enumerate(SITES18):
        prof[s] = [float(roc_auc_score(y, np.concatenate([S_PT[sd][g, j],
                                                          S_PT[sd][neg, j]])))
                   for sd in SEEDS]
    order = sorted(prof, key=lambda s: -np.mean(prof[s]))
    for s in order:
        m, lo, hi = t_ci(prof[s])
        run.log(f"    {s:<8} AUROC {m:.4f} [{lo:.4f},{hi:.4f}]"
                + (" ★1등" if s == order[0] else ""))
    return {"n": n, "prof": prof, "top": order[0]}

run.log("\n" + "=" * 112)
run.log("【P-4·P-5】 PTBDB 원문 표기별로 **어느 헤드가 1등인가**")
run.log("=" * 112)
run.log("  대조군은 **MI 가 전혀 없는 환자**로 공통 고정 — 모든 헤드를 같은 잣대로 잰다")
G_ANT = pure_group("anterior", "AMI")
G_INF = pure_group("inferior", "IMI")
HEADS = {"anterior(AMI 단독)": head_profile(G_ANT, "원문 'anterior' · AMI 단독"),
         "inferior(IMI 단독)": head_profile(G_INF, "원문 'inferior' · IMI 단독")}
HA, HI_ = HEADS["anterior(AMI 단독)"], HEADS["inferior(IMI 단독)"]

V = {}
def gap(h):
    return [h["prof"]["ASMI"][i] - h["prof"]["AMI"][i] for i in range(len(SEEDS))]

if HA and HA["n"] >= GMIN_SUB:
    m, lo, hi = t_ci(gap(HA))
    V["P-4"] = decide(lo, hi, HEAD_THR, ">")
    run.log(f"\n  P-4  'anterior' 에서 ASMI − AMI = {m:+.4f} [{lo:+.4f},{hi:+.4f}] "
            f"vs 문턱 {HEAD_THR} → {MARK[V['P-4']]}")
    run.log("       지지면 **PTBDB 의 'anterior' 는 우리 AMI 보다 ASMI 에 가깝다**")
else:
    V["P-4"] = None
    run.log(f"\n  P-4  'anterior·AMI 단독' 환자가 {GMIN_SUB}명 미만 → 채점 제외(미결)")

# ── P-5 이중차분: ASMI 헤드가 그냥 센 것이라면 'inferior' 에서도 같은 격차가 난다
if HA and HI_ and min(HA["n"], HI_["n"]) >= GMIN_SUB:
    did = [gap(HA)[i] - gap(HI_)[i] for i in range(len(SEEDS))]
    m5, lo5, hi5 = t_ci(did)
    V["P-5"] = decide(lo5, hi5, HEAD_THR, ">")
    run.log(f"  P-5  이중차분 (ASMI−AMI)_anterior − (ASMI−AMI)_inferior = "
            f"{m5:+.4f} [{lo5:+.4f},{hi5:+.4f}] → {MARK[V['P-5']]}")
    run.log(f"       참고: 'inferior' 1등 헤드 = {HI_['top']} "
            f"(IMI 여야 계측기가 정상이다)")
    run.log("       P-4 만 지지고 P-5 가 기각이면 '라벨 불일치' 가 아니라 "
            "**ASMI 헤드가 전반적으로 세다** 는 뜻이다 — 해석을 바꿔야 한다")
else:
    V["P-5"] = None
    run.log(f"  P-5  두 그룹 중 하나가 {GMIN_SUB}명 미만 → 채점 제외(미결)")


In [ ]:
# CELL 7 — 사전등록 채점
run.log("\n" + "=" * 112)
run.log("【사전등록 채점】 시드 t-CI · 환자 부트스트랩 CI 병기")
run.log("=" * 112)

sp7 = PL["7부위 OR"]
lo, hi = sp7["sp_bootci"]
V["P-1"] = decide(lo, hi, SP_THR, "<")
run.log(f"\n  P-1 7부위 OR 특이도 < {SP_THR}")
run.log(f"      Sp {sp7['sp']:.3f} 환자부트 [{lo:.3f},{hi:.3f}] → {MARK[V['P-1']]}")
run.log(f"      (같은 동작점에서 환자 단위 민감도 {sp7['se']:.3f} · PPV {sp7['ppv']:.3f})")

def or_seed(sd):
    a7 = float(ALARM[sd][:, VIEWS["7부위 OR"]].any(axis=1)[~IS_MI].mean())
    a6 = float(ALARM[sd][:, VIEWS["AMI 제외 6부위 OR"]].any(axis=1)[~IS_MI].mean())
    return (a7 / max(1 - a7, 1e-9)) / max(a6 / max(1 - a6, 1e-9), 1e-9)
ors = [or_seed(sd) for sd in SEEDS]
m, lo6, hi6 = t_ci(ors)
V["P-2"] = decide(lo6, hi6, OR_THR, ">")
run.log(f"\n  P-2 AMI 를 빼면 위양성 오즈비 >= {OR_THR}")
run.log(f"      오즈비 {m:.2f} [{lo6:.2f},{hi6:.2f}] → {MARK[V['P-2']]}")
run.log(f"      (참고 ΔSp {np.mean(PL['AMI 제외 6부위 OR']['sp_seeds']) - sp7['sp']:+.3f} — "
        "특이도가 0 근처면 이 값은 천장에 막힌다. 그래서 오즈비로 잰다)")
run.log("      지지면 경보율 58% 짜리 부위 하나가 OR 전체를 지배한다는 뜻이다")

m3, lo3, hi3 = t_ci(HIT["7부위 OR"]["partial"])
V["P-3"] = decide(lo3, hi3, HIT_THR, ">")
run.log(f"\n  P-3 MI 환자 부분적중 >= {HIT_THR}")
run.log(f"      {m3:.3f} [{lo3:.3f},{hi3:.3f}] → {MARK[V['P-3']]}")

run.log("\n" + "=" * 112)
for k in ("P-1", "P-2", "P-3", "P-4", "P-5"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
run.log("  ⚠️ 임계값은 내부에서 동결한 것이다 — 외부에서 재조정하면 이 숫자는 무의미해진다")
run.log("  ⚠️ 환자 290명 · 부위별 양성 1~44명. 부분군 " + str(GMIN_SUB) + "명 미만은 채점에서 뺐다")


In [ ]:
# CELL 8 — 그림
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))

names = list(VIEWS)
x = np.arange(len(names)); w = 0.28
for i, (k, lab) in enumerate([("se", "민감도"), ("sp", "특이도"), ("ppv", "PPV")]):
    ax[0].bar(x + (i - 1) * w, [PL[n][k] for n in names], w, label=lab)
ax[0].set_xticks(x); ax[0].set_xticklabels(names, fontsize=9)
ax[0].axhline(SP_THR, ls=":", c="k", lw=1)
ax[0].set_ylim(0, 1); ax[0].set_title(f"환자 단위 · P-1 {MARK[V.get('P-1')]}")
ax[0].legend(fontsize=8)

ax[1].bar(["독립 가정 상한", "실측(7부위 OR)"], [indep, 1 - sp7["sp"]],
          color=["#bbbbbb", "#d62728"])
ax[1].set_ylim(0, 1); ax[1].set_ylabel("비MI 환자 중 경보 비율")
ax[1].set_title("사후 유도의 독립 가정은 얼마나 틀렸나")

h = HEADS.get("anterior(AMI 단독)")
if h:
    ss = sorted(h["prof"], key=lambda s: -np.mean(h["prof"][s]))
    ax[2].bar(range(len(ss)), [np.mean(h["prof"][s]) for s in ss],
              color=["#2ca02c" if s == "ASMI" else "#ff7f0e" if s == "AMI" else "#cccccc"
                     for s in ss])
    ax[2].set_xticks(range(len(ss))); ax[2].set_xticklabels(ss, rotation=45, fontsize=9)
    ax[2].axhline(0.5, ls=":", c="k", lw=1); ax[2].set_ylim(0, 1)
    ax[2].set_ylabel("AUROC (vs 비MI)")
    ax[2].set_title(f"PTBDB 'anterior'(AMI 단독) n={h['n']} · P-4 {MARK[V.get('P-4')]}")
plt.tight_layout()
run.save_fig("exp20b_patient_level", fig)
plt.show()


In [ ]:
# CELL 9 — 결과 저장
res = {
    "week": 2, "exp_id": "exp20b_patient", "quest": "ailab-2026-0015",
    "step": "exp20b-patient-level", "split": "inter",
    "task": "환자 단위 집계 + AMI 라벨 정합 직접 검정(학습 0회)",
    "notebook": "notebooks/exp20b_patient_level.ipynb",
    "metric": "patient_specificity_or7", "value": round(float(sp7["sp"]), 4),
    "passed": bool(V.get("P-4") is True),
    "n_patients": len(PTS), "n_mi": int(IS_MI.sum()),
    "sites": SITES18, "deploy": DEPLOY, "seeds": SEEDS,
    "sens_target": SENS_TARGET, "gmin_subgroup": GMIN_SUB,
    "patient_level": {k: {kk: vv for kk, vv in v.items() if kk != "cols"}
                      for k, v in PL.items()},
    "hit": HIT,
    "independence_upper_bound": indep,
    "fp_odds_7": float(o7), "fp_odds_6": float(o6),
    "fp_odds_ratio_drop_ami": float(o7 / max(o6, 1e-9)),
    "observed_alarm_rate_nonmi": float(1 - sp7["sp"]),
    "head_profiles": {k: (None if v is None else
                          {"n": v["n"], "top": v["top"],
                           "auroc": {s: float(np.mean(x)) for s, x in v["prof"].items()}})
                      for k, v in HEADS.items()},
    "verdicts": {k: V.get(k) for k in ("P-1", "P-2", "P-3", "P-4", "P-5")},
    "caveats": [
        "임계값은 내부(PTB-XL 5겹 OOF)에서 민감도 0.90 으로 동결 — 외부 재조정 없음",
        "환자 290명 · 부분군 20명 미만은 채점 제외(GMIN_SUB)",
        "헤드 간 비교는 AUROC 로만 한다 — 확률 교정이 헤드마다 다르다",
        "본 모델은 급성 심근경색이 아니라 PTB-XL 의 심전도 판독 라벨을 예측한다"],
}
run.save_json("result", res)
run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts", "n_patients")},
                 ensure_ascii=False, indent=2))
print("\n다음: python pipelines/ingest_run.py --results result.json "
      "--notebook notebooks/exp20b_patient_level.ipynb")
